# Gold Layer - Consumption Views

## Purpose
Create **business-friendly views** on top of gold tables for dashboard consumption and reporting.

## Views to Create

1. **vw_regional_operations_dashboard** - Unified operational health view (all metrics)
2. **vw_daily_incident_trends** - Incident tracking and trending
3. **vw_market_weather_correlation** - Market price vs weather risk analysis

## Why Views?

* **Simplify access:** Pre-join common patterns
* **Abstract complexity:** Hide normalization logic
* **Govern access:** Control what business users see
* **Performance:** Optimize for BI tool queries
* **Consistency:** Single source of truth for dashboards

In [0]:
print("✅ Setup complete - creating consumption views")

In [0]:
%sql
-- Unified operational health view combining all metrics
-- Purpose: Single view for executive dashboards showing daily operational status

CREATE OR REPLACE VIEW vattenfall_dev.gold.vw_regional_operations_dashboard AS
SELECT 
  -- Date & Region
  c.report_day,
  c.region,
  
  -- Composite Health Score
  c.health_score,
  c.operational_condition,
  c.requires_alert,
  
  -- Market Metrics
  c.avg_price_eur_mwh,
  m.price_volatility_stddev,
  m.high_price_hours,
  m.total_volume_mwh,
  
  -- Weather Metrics
  c.weather_risk_score,
  c.weather_risk_level,
  w.avg_temperature_c,
  w.avg_wind_speed_ms,
  w.total_precipitation_mm,
  
  -- Incident Metrics
  c.total_incident_count,
  c.total_critical_incidents,
  c.total_customers_affected,
  i.total_duration_minutes,
  ROUND(i.total_duration_minutes / NULLIF(c.total_incident_count, 0), 1) as avg_duration_per_incident,
  
  -- Normalized Scores (for drill-down)
  c.market_norm,
  c.weather_norm,
  c.incident_norm,
  c.risk_score
  
FROM vattenfall_dev.gold.gold_regional_condition_daily c

-- Join market details
LEFT JOIN vattenfall_dev.gold.gold_daily_market_summary m
  ON c.report_day = m.report_day AND c.region = m.region

-- Join weather details  
LEFT JOIN vattenfall_dev.gold.gold_weather_impact_summary w
  ON c.report_day = w.report_day AND c.region = w.region

-- Join incident aggregates (sum across severity bands)
LEFT JOIN (
  SELECT 
    event_day,
    region,
    SUM(total_duration_minutes) as total_duration_minutes
  FROM vattenfall_dev.gold.gold_grid_incident_summary
  GROUP BY event_day, region
) i
  ON c.report_day = i.event_day AND c.region = i.region

ORDER BY c.report_day DESC, c.region;

-- Verify
SELECT COUNT(*) as row_count FROM vattenfall_dev.gold.vw_regional_operations_dashboard;

In [0]:
%sql
-- Incident-focused view for operational teams
-- Purpose: Track incident patterns, severity, customer impact over time

CREATE OR REPLACE VIEW vattenfall_dev.gold.vw_daily_incident_trends AS
SELECT 
  event_day as report_day,
  region,
  severity_band,
  
  -- Incident Counts
  incident_count,
  elevated_incident_count,
  critical_incident_count,
  
  -- Duration Metrics
  total_duration_minutes,
  avg_duration_minutes,
  max_duration_minutes,
  
  -- Customer Impact
  total_customers_affected,
  avg_customers_per_incident,
  max_customers_affected,
  
  -- Event Diversity
  unique_event_types,
  
  -- Derived Metrics
  ROUND(total_customers_affected / NULLIF(incident_count, 0), 0) as customers_per_incident_check,
  ROUND(total_duration_minutes / NULLIF(incident_count, 0), 1) as duration_per_incident,
  
  -- Severity Flag
  CASE 
    WHEN critical_incident_count > 0 THEN 'CRITICAL_PRESENT'
    WHEN elevated_incident_count > 0 THEN 'ELEVATED_PRESENT'
    ELSE 'ROUTINE'
  END as incident_severity_flag,
  
  -- Day Categorization
  CASE 
    WHEN incident_count >= 6 THEN 'HIGH_VOLUME'
    WHEN incident_count >= 3 THEN 'MODERATE_VOLUME'
    WHEN incident_count >= 1 THEN 'LOW_VOLUME'
    ELSE 'NO_INCIDENTS'
  END as volume_category
  
FROM vattenfall_dev.gold.gold_grid_incident_summary
ORDER BY event_day DESC, region, severity_band;

-- Verify
SELECT COUNT(*) as row_count FROM vattenfall_dev.gold.vw_daily_incident_trends;

In [0]:
%sql
-- Market price vs weather risk correlation view
-- Purpose: Analyze relationship between weather conditions and electricity pricing

CREATE OR REPLACE VIEW vattenfall_dev.gold.vw_market_weather_correlation AS
SELECT 
  m.report_day,
  m.region,
  
  -- Market Metrics
  m.avg_price_eur_mwh,
  m.max_price_eur_mwh,
  m.min_price_eur_mwh,
  m.price_volatility_stddev,
  m.high_price_hours,
  
  -- Weather Metrics
  w.avg_temperature_c,
  w.avg_wind_speed_ms,
  w.total_precipitation_mm,
  w.avg_cloud_cover_percent,
  w.weather_risk_score,
  w.weather_risk_level,
  
  -- Price Category
  CASE 
    WHEN m.avg_price_eur_mwh >= 50 THEN 'HIGH_PRICE'
    WHEN m.avg_price_eur_mwh >= 45 THEN 'MODERATE_PRICE'
    ELSE 'LOW_PRICE'
  END as price_category,
  
  -- Temperature Category
  CASE 
    WHEN w.avg_temperature_c <= -2 THEN 'EXTREME_COLD'
    WHEN w.avg_temperature_c <= 0 THEN 'FREEZING'
    WHEN w.avg_temperature_c <= 5 THEN 'COLD'
    ELSE 'MODERATE'
  END as temp_category,
  
  -- Wind Category
  CASE 
    WHEN w.avg_wind_speed_ms >= 10 THEN 'HIGH_WIND'
    WHEN w.avg_wind_speed_ms >= 8 THEN 'MODERATE_WIND'
    ELSE 'LOW_WIND'
  END as wind_category,
  
  -- Combined Risk Flag
  CASE 
    WHEN w.weather_risk_level IN ('EXTREME', 'HIGH') AND m.avg_price_eur_mwh >= 50 THEN 'CRITICAL_COMBO'
    WHEN w.weather_risk_level IN ('EXTREME', 'HIGH') OR m.avg_price_eur_mwh >= 50 THEN 'ELEVATED_RISK'
    ELSE 'NORMAL'
  END as combined_risk_flag
  
FROM vattenfall_dev.gold.gold_daily_market_summary m

INNER JOIN vattenfall_dev.gold.gold_weather_impact_summary w
  ON m.report_day = w.report_day AND m.region = w.region
  
ORDER BY m.report_day DESC, m.region;

-- Verify
SELECT COUNT(*) as row_count FROM vattenfall_dev.gold.vw_market_weather_correlation;

In [0]:
%sql
-- Summary of all views created
SELECT 
  'vw_regional_operations_dashboard' as view_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT report_day) as unique_days,
  COUNT(DISTINCT region) as unique_regions
FROM vattenfall_dev.gold.vw_regional_operations_dashboard

UNION ALL

SELECT 
  'vw_daily_incident_trends' as view_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT report_day) as unique_days,
  COUNT(DISTINCT region) as unique_regions
FROM vattenfall_dev.gold.vw_daily_incident_trends

UNION ALL

SELECT 
  'vw_market_weather_correlation' as view_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT report_day) as unique_days,
  COUNT(DISTINCT region) as unique_regions
FROM vattenfall_dev.gold.vw_market_weather_correlation

ORDER BY view_name;

---

## ✅ Views Created

### 1. vw_regional_operations_dashboard
**Purpose:** Executive dashboard - unified operational health

**Grain:** Daily × Region (60 records)

**Key Columns:**
* Composite health score & operational condition
* Market: price, volatility, high-price hours, volume
* Weather: risk score, temperature, wind, precipitation
* Incidents: count, critical count, customers affected, duration
* Normalized scores for drill-down analysis

**Use Case:** Power BI/Tableau executive dashboards showing daily operational status

---

### 2. vw_daily_incident_trends
**Purpose:** Operational teams - incident tracking

**Grain:** Daily × Region × Severity Band (97 records)

**Key Columns:**
* Incident counts by severity
* Duration metrics (total, avg, max)
* Customer impact (total, avg, max)
* Derived metrics: customers per incident, duration per incident
* Volume category (HIGH/MODERATE/LOW)
* Severity flag (CRITICAL_PRESENT/ELEVATED_PRESENT/ROUTINE)

**Use Case:** Operations center dashboards, incident response tracking, trend analysis

---

### 3. vw_market_weather_correlation
**Purpose:** Analytics - price vs weather relationship

**Grain:** Daily × Region (60 records)

**Key Columns:**
* Market: price metrics, volatility, high-price hours
* Weather: temperature, wind, precipitation, risk score
* Categorizations: price category, temp category, wind category
* Combined risk flag (CRITICAL_COMBO/ELEVATED_RISK/NORMAL)

**Use Case:** Data science analysis, forecasting models, correlation studies

---

## View Benefits

✅ **Simplified Queries:** Pre-joined, no need to understand table relationships

✅ **Consistent Logic:** Categorizations applied uniformly

✅ **Performance:** Optimized for BI tool consumption

✅ **Governance:** Can grant view access without exposing base tables

✅ **Abstraction:** Business users see metrics, not normalization logic

---